# Using Spark SQL

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

conf = SparkConf()

conf.setAppName("Spark SQL Example") # Name of spark application, useful for logs
conf.set("spark.hadoop.fs.s3a.endpoint","http://minio:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", "admin") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", "@admin123") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## Create Data Frame

In [3]:
data = [("Product A", 300, "Tech"),
               ("Product B", 250, "Sport"),
               ("Product C", 150, "Book")
              ]

schema = StructType([
        StructField("product_name", StringType(), True),
        StructField("price", IntegerType(), True),
        StructField("departament", StringType(), True)
        ])

data_frame = spark.createDataFrame(data= data, schema= schema)
data_frame.show()

+------------+-----+-----------+
|product_name|price|departament|
+------------+-----+-----------+
|   Product A|  300|       Tech|
|   Product B|  250|      Sport|
|   Product C|  150|       Book|
+------------+-----+-----------+



## Create a Tempo View to use SQL

In [4]:
data_frame.createOrReplaceTempView("table_example")

## Creating a Spark SQL

In [7]:
query = spark.sql("""
        select * from table_example
        where price >= 250
""")

query.show()

+------------+-----+-----------+
|product_name|price|departament|
+------------+-----+-----------+
|   Product A|  300|       Tech|
|   Product B|  250|      Sport|
+------------+-----+-----------+



In [8]:
query.write.format("delta").mode("append").save("s3a://bronze/table_spark_sql")